In [23]:
from langgraph.graph import START, END, MessagesState, StateGraph
from langgraph.types import Command, interrupt
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.prebuilt import ToolNode
from langchain_core.tools import tool
from typing import Literal
from langchain_core.messages import HumanMessage, AIMessage

In [12]:
@tool 
def book_flight(destination: str):
    """ books flight """
    print(f"Flight booked to {destination}")
    return f"Flight booked to {destination}"

In [28]:
def review(state: MessagesState) -> Command[Literal["tools", END]]:
    last = state["messages"][-1]
    print(state["messages"][-1])
    decision = interrupt(
        {
            "question": "Approve this booking?",
            "tool_call": last.tool_calls,
        }
    )

    if decision == "approve":
        return Command(goto="tools")

    return Command(goto=END)


In [29]:
# agent
def agent(state: MessagesState):
    return {
        "messages": [
            AIMessage(
                content="I will book your flight.",
                tool_calls=[
                    {
                    "id": "1",
                    "name": "book_flight",
                    "args": {
                        "destination": "Tokyo"                        
                    }
                    }
                ],
            )
        ]
    }
   

In [30]:
def tools(state: MessagesState):
    last = state["messages"][-1]

    destination = last.tool_calls[0]["args"]["destination"]

    result = book_flight.invoke(
        {"destination": destination}
    )

    return {
        "messages": [
            AIMessage(content=result)
        ]
    }

In [31]:
builder = StateGraph(MessagesState)

builder.add_node("agent", agent)
builder.add_node("review", review)
builder.add_node("tools", tools)

builder.add_edge(START, "agent")
builder.add_edge("agent", "review")
builder.add_edge("tools", END)

In [32]:
graph = builder.compile(
    checkpointer=InMemorySaver()
)

In [33]:
config = {
    "configurable": {
        "thread_id": "flight-1"
    }
}

result = graph.invoke(
    {
        "messages": [
            HumanMessage(content="Book a flight to Tokyo")
        ]
    },
    config=config,
)

print(result["__interrupt__"])

content='I will book your flight.' additional_kwargs={} response_metadata={} id='298c01c4-6d68-4335-b6f4-1d0ce5d837e4' tool_calls=[{'name': 'book_flight', 'args': {'destination': 'Tokyo'}, 'id': '1', 'type': 'tool_call'}] invalid_tool_calls=[]
[Interrupt(value={'question': 'Approve this booking?', 'tool_call': [{'name': 'book_flight', 'args': {'destination': 'Tokyo'}, 'id': '1', 'type': 'tool_call'}]}, id='4140c814b82d7770cee58cff7ca867d3')]


In [34]:
graph.invoke(
    Command(resume="approve"),
    config=config,
)

content='I will book your flight.' additional_kwargs={} response_metadata={} id='298c01c4-6d68-4335-b6f4-1d0ce5d837e4' tool_calls=[{'name': 'book_flight', 'args': {'destination': 'Tokyo'}, 'id': '1', 'type': 'tool_call'}] invalid_tool_calls=[]
Flight booked to Tokyo


{'messages': [HumanMessage(content='Book a flight to Tokyo', additional_kwargs={}, response_metadata={}, id='5bfb6ed5-7676-4039-8cc7-3b837c4a51a4'),
  AIMessage(content='I will book your flight.', additional_kwargs={}, response_metadata={}, id='298c01c4-6d68-4335-b6f4-1d0ce5d837e4', tool_calls=[{'name': 'book_flight', 'args': {'destination': 'Tokyo'}, 'id': '1', 'type': 'tool_call'}], invalid_tool_calls=[]),
  AIMessage(content='Flight booked to Tokyo', additional_kwargs={}, response_metadata={}, id='b76bc262-41ef-464b-ab0c-b0201cd42022', tool_calls=[], invalid_tool_calls=[])]}

Approve/ reject/ edit a tool

In [35]:
from typing import Literal, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, interrupt

In [37]:
def	review_tool_call(state:	MessagesState)	->	Command [Literal["tools",	"agent"]]:
    last	=	state["messages"][-1]
    decision	=	interrupt({
        "question":	"Approve	this	tool	call?",
        "tool_calls":	last.tool_calls,
})
    
    if	decision["type"]	==	"approve":
        return	Command(goto="tools")
    if	decision["type"]	==	"edit":
        last.tool_calls[0]["args"]	=	decision["args"]
        return	Command(update={"messages":	[last]},	goto="tools")
#	reject
    return	Command(
        update={"messages":	[{"role":	"tool",	"tool_call_id":	last.tool_calls[0]["id"], "content":	"Rejected	by	human."}]}, goto="agent",)

In [38]:
builder.add_node("review",	review_tool_call)
builder.add_edge("agent",	"review")			#	instead	of	agent	->	tools	directly

Adding a node to a graph that has already been compiled. This will not be reflected in the compiled graph.


ValueError: Node `review` already present.

In [ ]:
graph.invoke(Command(resume={"type":	"approve"}),	config)
graph.invoke(Command(resume={"type":	"edit",	"args":	{"city":	"Osaka"}}),	config)
graph.invoke(Command(resume={"type":	"reject"}),	config)